# LAB: Demo phân loại hoa Iris bằng Streamlit

Notebook này trình bày các bước nạp dữ liệu, đánh giá Gaussian Naive Bayes, huấn luyện mô hình cuối cùng trên toàn bộ dữ liệu và kiểm tra bốn trường hợp mẫu. Ứng dụng giao diện được đặt riêng trong `app.py` để dễ chạy trên máy cá nhân hoặc triển khai qua Streamlit Community Cloud.

GitHub: https://github.com/TranMinhHuy15/AIL_IRIS_Classification

## 1. Nạp thư viện và dữ liệu Iris

In [ ]:
# Import thư viện và nạp bộ dữ liệu Iris có sẵn trong scikit-learn.
import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.datasets import load_iris
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import GaussianNB

iris = load_iris()
X = iris.data
y = iris.target
feature_names = iris.feature_names
target_names = iris.target_names

print(f'Số mẫu: {X.shape[0]}')
print(f'Số đặc trưng: {X.shape[1]}')
print(f'Các loài: {list(target_names)}')

In [ ]:
# Tạo bảng để xem các đặc trưng và nhãn loài hoa.
iris_df = pd.DataFrame(X, columns=feature_names)
iris_df['Loài hoa'] = [target_names[index].capitalize() for index in y]
display(iris_df.head())
print('Số mẫu theo loài:')
display(iris_df['Loài hoa'].value_counts().rename_axis('Loài').to_frame('Số mẫu'))

## 2. Đánh giá trên tập Test 80/20

In [ ]:
# Chia tập có phân tầng để giữ tỷ lệ ba loài trong cả Train và Test.
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Huấn luyện mô hình đánh giá trên tập Train và chấm điểm trên tập Test.
evaluation_model = GaussianNB()
evaluation_model.fit(X_train, y_train)
test_predictions = evaluation_model.predict(X_test)
test_accuracy = accuracy_score(y_test, test_predictions)
print(f'Accuracy trên tập Test: {test_accuracy:.2%}')

## 3. Huấn luyện mô hình cuối trên toàn bộ dữ liệu

In [ ]:
# Mô hình dùng cho ứng dụng được fit trên toàn bộ 150 mẫu như yêu cầu bài lab.
model = GaussianNB()
model.fit(X, y)
print('Đã huấn luyện Gaussian Naive Bayes trên toàn bộ dữ liệu Iris.')

In [ ]:
# Tạo hàm dự đoán để tách phần xử lý mô hình khỏi phần trình bày.
def predict_iris(features):
    input_array = np.asarray(features, dtype=float).reshape(1, -1)
    predicted_index = int(model.predict(input_array)[0])
    probabilities = model.predict_proba(input_array)[0]
    return target_names[predicted_index].capitalize(), probabilities

# Bốn trường hợp theo đề: ba loài điển hình và một điểm nằm giữa hai loài.
test_cases = {
    'Setosa': [5.1, 3.5, 1.4, 0.2],
    'Versicolor': [6.0, 2.9, 4.5, 1.5],
    'Virginica': [6.5, 3.0, 5.8, 2.2],
    'Điểm giữa hai loài': [6.0, 2.7, 5.0, 1.7],
}

# Gom dự đoán và xác suất của cả ba lớp thành bảng để dễ đối chiếu.
rows = []
for case_name, values in test_cases.items():
    predicted_species, probabilities = predict_iris(values)
    rows.append({
        'Trường hợp': case_name,
        'Đầu vào (SL, SW, PL, PW)': values,
        'Dự đoán': predicted_species,
        **{f'P({name.capitalize()})': float(probability) for name, probability in zip(target_names, probabilities)},
    })

results_df = pd.DataFrame(rows)
display(results_df.style.format({f'P({name.capitalize()})': '{:.2%}' for name in target_names}))

## 4. Chạy ứng dụng Streamlit

Mã giao diện đầy đủ nằm trong `app.py`; file này dùng `@st.cache_resource` để mô hình không bị huấn luyện lại khi Streamlit chạy lại giao diện. Chạy trên máy cá nhân từ thư mục dự án bằng `streamlit run app.py`. Link ứng dụng công khai sẽ được ghi trong README sau khi triển khai trên Streamlit Community Cloud.

Kết quả điểm giữa hai loài cần được nhận xét qua ba xác suất: mô hình chọn lớp có xác suất lớn nhất, còn các xác suất gần nhau biểu thị trường hợp ranh giới khó phân biệt.